# Comparação 3 — backlash nativo do ROSS

Este notebook ativa `backlash` no próprio `MultiRotor` do ROSS. Nesse caminho, o ROSS usa internamente a estratégia equivalente a `lambda K0: K0`.

## 1. Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ross as rs
ROOT=Path(r'C:\Users\M\Desktop\doutorado\teste_backlash_ross')

## 2. Modelagem do multirrotor com backlash nativo

In [ ]:
z=20; module=0.01
steel=rs.Material(name='Steel',rho=7850,E=2e11,Poisson=0.3)
stiff=rs.Material(name='Steel_Stiff',rho=0.01,E=1e15,Poisson=0.3)
shaft=[rs.ShaftElement(L=0.0001,idl=0.0,odl=0.0001,material=stiff,n=0)]
bearing=rs.BearingElement(n=0,kxx=1e8,kyy=1e8,cxx=512.64,cyy=512.64)
bore=np.sqrt((module*z)**2-4*6.57/(np.pi*0.03*steel.rho))
gear=rs.GearElementTVMS(n=0,material=steel,width=0.03,bore_diameter=bore,module=module,n_teeth=z,pr_angle=np.radians(20),helix_angle=0,addendum_coeff=1,tip_clearance_coeff=0.25)
gear.m=6.57; gear.Ip=0.0365; gear.Id=0.0001*0.0365/2
rotor=rs.Rotor(shaft_elements=shaft,disk_elements=[gear],bearing_elements=[bearing])
omega=4500*np.pi/30
model=rs.MultiRotor(driving_rotor=rotor,driven_rotor=rotor,coupled_nodes=(0,0),update_mesh_stiffness=True,
    square_varying_stiffness={'enable':True,'amplitude_ratio':0.275},orientation_angle=0.0,position='above',
    backlash={'enable':True,'initial_value':50e-6,'error_amp':20e-6,'smooth_operator':False,'sigma':1e5})
print('backlash nativo:',model.mesh.backlash is not None)
print('função de rigidez:',model.add_coupling_stiffness)

## 3. Comparação nos DOFs das engrenagens

In [ ]:
gear_dofs=list(model.mesh.driving_gear.dof_global_index.values())+list(model.mesh.driven_gear.dof_global_index.values())
K=model.K(omega)
K0=model._join_matrices(model.rotors['driving'].K(omega,omega),model.rotors['driven'].K(omega,omega*model.mesh.gear_ratio))
Kgear=K[np.ix_(gear_dofs,gear_dofs)]
K0gear=K0[np.ix_(gear_dofs,gear_dofs)]
Kcoupling=model.K_coupling*model.mesh.stiffness
print('DOFs:',gear_dofs)
print('mesh.stiffness =',model.mesh.stiffness)
print('norma Kgear-K0gear =',np.linalg.norm(Kgear-K0gear))
print('norma coupling esperado =',np.linalg.norm(Kcoupling[np.ix_(gear_dofs,gear_dofs)]))

## 4. Valores e teste da estratégia lambda K0

In [ ]:
display(pd.DataFrame(Kgear,index=gear_dofs,columns=gear_dofs))
print('Kgear - K0gear:')
display(pd.DataFrame(Kgear-K0gear,index=gear_dofs,columns=gear_dofs))
np.testing.assert_allclose(Kgear,K0gear,rtol=0,atol=0)
print('Resultado: o backlash nativo mantém mesh.stiffness para a força, mas não soma K_coupling em K.')

## 5. Visualização da contribuição linear

In [ ]:
fig,ax=plt.subplots(figsize=(9,7))
im=ax.imshow(Kgear-K0gear,cmap='coolwarm')
ax.set_title('ROSS nativo: Kgear - K0gear')
fig.colorbar(im,ax=ax,label='Rigidez')
plt.show()